# 01 — Exploratory Data Analysis

Implements **plan §5**. EDA on this dataset is constrained by the PCA anonymization,
so the goal is less about discovering domain meaning and more about characterizing
distributions, leakage risks, and the imbalance regime.

**Outputs of this notebook (§5.5):**
- A list of features to log-transform (Amount and time-derived).
- A short list of likely top-importance features as a sanity check for later.
- A duplicate-handling decision.
- A clear statement of the imbalance ratio for class-weighting later.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# So `from src.X import ...` works whether the notebook is run from the repo
# root (docker-compose default) or from inside notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import DEFAULT_DATA_PATH, FEATURE_COLS, TARGET_COL, load_raw, sha256_of_file

sns.set_theme(context='notebook', style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda v: f'{v:.6g}')

## 1. Load and verify (§5.1 data quality checks)

In [ ]:
data_path = ROOT / DEFAULT_DATA_PATH
print(f'Reading {data_path} ({data_path.stat().st_size / 1e6:.1f} MB)')
print(f'SHA-256: {sha256_of_file(data_path)}')

# Load WITHOUT dropping duplicates first so we can document the decision.
df_raw = load_raw(data_path, drop_duplicates=False)
df_raw.shape, df_raw[TARGET_COL].sum()

In [ ]:
# Schema, null check, duplicates, time monotonicity.
print('Nulls per column (any?):', df_raw.isna().sum().sum())
print('Exact duplicates:', int(df_raw.duplicated().sum()))
print('Time is monotonic non-decreasing:',
      bool(df_raw["Time"].is_monotonic_increasing))

# Decision: drop exact duplicates. Stratified splitting on duplicates inflates
# leakage risk between train and test (same row in both splits).
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f'After dedup: {len(df):,} rows')

## 2. Class imbalance (§5.4)

In [ ]:
n_total = len(df)
n_fraud = int(df[TARGET_COL].sum())
n_genuine = n_total - n_fraud
fraud_rate = n_fraud / n_total
imb_ratio = n_genuine / n_fraud

print(f'Total rows:    {n_total:>10,}')
print(f'Genuine:       {n_genuine:>10,} ({(1 - fraud_rate) * 100:.4f}%)')
print(f'Fraud:         {n_fraud:>10,} ({fraud_rate * 100:.4f}%)')
print(f'Imbalance:     1 fraud per {imb_ratio:.0f} genuine')
print(f'Naive accuracy (predict all genuine): {(1 - fraud_rate) * 100:.4f}%')
print('=> accuracy is meaningless here; use PR-AUC and recall@FPR instead.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh([0], [n_genuine], color='steelblue', label=f'Genuine ({n_genuine:,})')
ax.barh([0], [n_fraud], left=[n_genuine], color='crimson',
        label=f'Fraud ({n_fraud:,})')
ax.set_yticks([])
ax.set_xlabel('Transactions')
ax.set_title(f'Class imbalance: fraud is {fraud_rate * 100:.3f}% of all rows')
ax.legend(loc='center')
plt.tight_layout()
plt.show()

## 3. Univariate analysis (§5.2)

In [ ]:
# Amount: right-skewed, motivates log1p transform.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df['Amount'], bins=80, color='steelblue')
axes[0].set_title('Amount (linear)')
axes[0].set_xlabel('Amount')
axes[1].hist(np.log1p(df['Amount']), bins=80, color='steelblue')
axes[1].set_title('log1p(Amount)')
axes[1].set_xlabel('log1p(Amount)')
plt.tight_layout()
plt.show()

df['Amount'].describe()

In [ ]:
# Time -> hour-of-day. Plot fraud rate by hour.
df_t = df.copy()
df_t['hour'] = ((df_t['Time'] // 3600) % 24).astype(int)
by_hour = df_t.groupby('hour').agg(
    n=('Class', 'size'),
    fraud=('Class', 'sum'),
)
by_hour['fraud_rate'] = by_hour['fraud'] / by_hour['n']

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(by_hour.index, by_hour['fraud_rate'] * 100, color='crimson')
ax.set_xlabel('Hour of day (relative to first transaction)')
ax.set_ylabel('Fraud rate (%)')
ax.set_title('Fraud rate by hour-of-day')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()
by_hour

In [ ]:
# Per-feature histograms of V1..V28 overlaid by class. Identify which features
# show visible class separation -- typically V14, V12, V10, V17, V4 per the plan.
v_cols = [f'V{i}' for i in range(1, 29)]
fig, axes = plt.subplots(7, 4, figsize=(14, 18))
for ax, col in zip(axes.ravel(), v_cols):
    ax.hist(df.loc[df[TARGET_COL] == 0, col], bins=80, alpha=0.5,
            label='genuine', density=True, color='steelblue')
    ax.hist(df.loc[df[TARGET_COL] == 1, col], bins=80, alpha=0.7,
            label='fraud', density=True, color='crimson')
    ax.set_title(col, fontsize=9)
    ax.set_yticks([])
axes[0, 0].legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()

## 4. Bivariate / class analysis (§5.3)

In [ ]:
# Univariate class separation via mean-difference / |t-stat|-style ranking.
# Cheap, interpretable, and a useful sanity check against later SHAP rankings.
rank_rows = []
for col in v_cols + ['Amount', 'Time']:
    g = df.loc[df[TARGET_COL] == 0, col]
    f = df.loc[df[TARGET_COL] == 1, col]
    pooled = (g.var(ddof=1) / len(g) + f.var(ddof=1) / len(f)) ** 0.5
    if pooled == 0:
        score = 0.0
    else:
        score = abs(g.mean() - f.mean()) / pooled
    rank_rows.append({'feature': col, 'separation_score': float(score)})
rank = (pd.DataFrame(rank_rows)
        .sort_values('separation_score', ascending=False)
        .reset_index(drop=True))
rank.head(12)

In [ ]:
# Boxplots of the top-6 separating features.
top6 = rank.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), top6):
    sns.boxplot(data=df, x=TARGET_COL, y=col, ax=ax, showfliers=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap. V-features should be uncorrelated (PCA output) but we
# expect Amount to correlate with a handful of V-features.
corr = df[FEATURE_COLS + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Feature correlations (Pearson)')
plt.tight_layout()
plt.show()

## 5. Outputs of EDA stage (§5.5)

Recorded here for downstream notebooks and the writeup:

1. **Log-transform list:** `Amount` (heavy right tail). Time-derived features
   are bounded so no log transform needed; tree models won't benefit anyway.
2. **Likely top-importance features (sanity check for later SHAP):** the top of
   the separation-score table above. The plan predicts V14, V12, V10, V17, V4
   will dominate; verify after training.
3. **Duplicate-handling decision:** drop exact duplicates (~1k rows). Reasoning:
   stratified splitting on duplicates risks the same row appearing in train and
   test, which leaks information.
4. **Imbalance ratio:** computed above and recorded as `n_genuine / n_fraud`.
   Used directly as `scale_pos_weight` in XGBoost (plan §8 "class weights").